# Lakehouse Analysis with DuckDB

This notebook demonstrates how to query Parquet data stored in MinIO using DuckDB.

## Setup

1. Ensure MinIO is running: `docker compose up -d minio` (in WorkFlow directory)
2. Ensure data has been processed by DAG `D2_Process_HTML_to_Lakehouse`
3. Install dependencies: `pip install duckdb pandas pyarrow minio`

## Data Flow

```
Local HTML Files → DAG Processing → MinIO (raw + staging) → DuckDB Queries
```



In [ ]:
# Import required libraries
import sys
import os
from datetime import datetime

# Add Lib to path if needed
if '/opt/airflow/lib' not in sys.path:
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Lib'))

from duckdb_client import create_duckdb_client
from parquet_io import load_dataframe_from_minio
import pandas as pd

print("✅ Libraries imported successfully")



## Method 1: Direct DataFrame Loading

Load Parquet files directly from MinIO as pandas DataFrames (simpler, but loads all data into memory).


In [ ]:
# Load DataFrame directly from MinIO
# Replace date with your actual processing date
date = "2024-12-11"  # Update this to match your DAG execution date

df = load_dataframe_from_minio(
    zone="staging",
    table_name="job_table",
    date=date,
    filename="jobs.parquet"
)

if df is not None:
    print(f"✅ Loaded {len(df)} rows")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst few rows:")
    display(df.head())
else:
    print("❌ Failed to load data. Make sure DAG has been executed and data exists in MinIO.")



## Method 2: DuckDB SQL Queries

Use DuckDB to query Parquet files with SQL (more efficient for large datasets, supports filtering/aggregation before loading).


In [ ]:
# Create DuckDB client
client = create_duckdb_client()

# Query single Parquet file
date = "2024-12-11"  # Update this to match your DAG execution date

df_query = client.query_parquet_file(
    zone="staging",
    table_name="job_table",
    date=date,
    filename="jobs.parquet",
    sql_query="""
    SELECT 
        job_id,
        head as job_title,
        head_url as job_url,
        info,
        processed_date
    FROM parquet_file
    WHERE head IS NOT NULL
    LIMIT 10
    """
)

if df_query is not None:
    print(f"✅ Query returned {len(df_query)} rows")
    display(df_query)
else:
    print("❌ Query failed. Check that data exists in MinIO.")

# Clean up
client.close()



## Method 3: Register Table and Query Multiple Times

Register a Parquet file as a table, then run multiple queries efficiently.


In [ ]:
# Create client with context manager (auto cleanup)
date = "2024-12-11"  # Update this to match your DAG execution date

with create_duckdb_client() as client:
    # Register Parquet file as a table
    success = client.register_parquet_table(
        table_name="jobs",
        zone="staging",
        table_name_path="job_table",
        date=date,
        filename="jobs.parquet"
    )
    
    if success:
        # Query 1: Count total jobs
        df_count = client.query_sql("SELECT COUNT(*) as total_jobs FROM jobs")
        print("Total jobs:")
        display(df_count)
        
        # Query 2: Find jobs with specific keywords
        df_keywords = client.query_sql("""
            SELECT 
                job_id,
                head as job_title,
                head_url
            FROM jobs
            WHERE head LIKE '%Engineer%' OR head LIKE '%Data%'
            LIMIT 10
        """)
        print("\nJobs with 'Engineer' or 'Data' in title:")
        display(df_keywords)
        
        # Query 3: Group by and aggregate
        df_stats = client.query_sql("""
            SELECT 
                COUNT(*) as total,
                COUNT(DISTINCT head) as unique_titles,
                COUNT(DISTINCT job_id) as unique_jobs
            FROM jobs
        """)
        print("\nStatistics:")
        display(df_stats)
    else:
        print("❌ Failed to register table")



## Method 4: Query Multiple Files (Date Range)

Query all Parquet files in a prefix (e.g., all dates or multiple partitions).


In [ ]:
# Query all files in a prefix (useful for date ranges or multiple partitions)
with create_duckdb_client() as client:
    # Query all Parquet files in the staging/job_table prefix
    df_all = client.query_parquet_prefix(
        zone="staging",
        table_name="job_table",
        date="2024-12-11",  # Or None to query all dates
        sql_query="""
        SELECT 
            COUNT(*) as total_records,
            COUNT(DISTINCT job_id) as unique_jobs,
            MIN(processed_date) as earliest_date,
            MAX(processed_date) as latest_date
        FROM parquet_files
        """
    )
    
    if df_all is not None:
        print("Aggregated statistics across all files:")
        display(df_all)
    else:
        print("❌ Query failed or no files found")



## Example: Advanced Analysis

Perform more complex analysis using DuckDB's SQL capabilities.


In [ ]:
# Advanced analysis example
date = "2024-12-11"  # Update this to match your DAG execution date

with create_duckdb_client() as client:
    # Register table
    client.register_parquet_table(
        table_name="jobs",
        zone="staging",
        table_name_path="job_table",
        date=date,
        filename="jobs.parquet"
    )
    
    # Example: Analyze job titles
    df_analysis = client.query_sql("""
        SELECT 
            head as job_title,
            COUNT(*) as count,
            COUNT(DISTINCT job_id) as unique_jobs
        FROM jobs
        WHERE head IS NOT NULL
        GROUP BY head
        ORDER BY count DESC
        LIMIT 20
    """)
    
    print("Top job titles:")
    display(df_analysis)
    
    # Example: Check data quality
    df_quality = client.query_sql("""
        SELECT 
            COUNT(*) as total_rows,
            COUNT(head) as has_title,
            COUNT(head_url) as has_url,
            COUNT(info) as has_info,
            COUNT(job_details_content) as has_details,
            COUNT(company_box_content) as has_company_info
        FROM jobs
    """)
    
    print("\nData quality metrics:")
    display(df_quality)

